In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [11]:
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader


class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted([
            'blues', 'classical', 'country', 'disco', 'hiphop',
            'jazz', 'metal', 'pop', 'reggae', 'rock'
        ])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
    
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]
    
        feature = torch.load(file_path)
    
        # convert stereo -> mono if needed
        if feature.shape[0] > 1:
            feature = feature.mean(dim=0, keepdim=True)
    
        return feature, label


class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CRNN, self).__init__()

        # ----- CNN (Feature Extractor) -----
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # After two pooling layers:
        # Mel bins: 128 -> 64 -> 32
        # Channels: 64
        # Flattened feature size = 64 * 32 = 2048

        # ----- RNN -----
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # ----- Final Classifier -----
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        # x shape: (Batch, 1, 128, Time)

        # CNN feature extraction
        x = self.cnn(x)
        # shape: (Batch, 64, 32, Time)

        b, c, f, t = x.shape

        # ----- Bridge (reshape for LSTM) -----
        x = x.permute(0, 3, 1, 2)       # (Batch, Time, Channels, Mels)
        x = x.reshape(b, t, c * f)      # (Batch, Time, 2048)

        # ----- RNN -----
        x, _ = self.lstm(x)             # (Batch, Time, 128)

        # ----- Global Max Pool over time -----
        x, _ = torch.max(x, dim=1)      # (Batch, 128)

        # ----- Final prediction -----
        logits = self.fc(x)

        return logits

In [3]:
import glob
import os

base_dir = "/kaggle/working/synthetic_mashups/train"

wav_files = glob.glob(os.path.join(base_dir, "**", "*.wav"), recursive=True)

print(len(wav_files))

500


In [4]:
import torchaudio
import glob
import os

# path to generated mashups
base_dir = "/kaggle/working/synthetic_mashups/train"

# pick any wav file
wav_file = glob.glob(os.path.join(base_dir, "**", "*.wav"), recursive=True)[0]

# load audio
waveform, sr = torchaudio.load(wav_file)

# print tensor shape
print(tuple(waveform.shape))

(2, 661500)


In [5]:
import torch
import glob
import os

features_dir = "/kaggle/working/features/train"

# get any feature file
pt_file = glob.glob(os.path.join(features_dir, "**", "*.pt"), recursive=True)[0]

# load tensor
feature = torch.load(pt_file)

# print tensor shape
print(tuple(feature.shape))

(2, 128, 1292)


In [12]:
import torch
import torch.nn as nn

# CNN backbone from the CRNN model
cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

# dummy batch: (batch_size=32, channels=1, mels=128, time=1293)
x = torch.randn(32, 1, 128, 1293)

# pass through CNN
out = cnn(x)

# print tensor shape
print(tuple(out.shape))

(32, 64, 32, 323)


In [13]:
# create model
model = CRNN()

# count trainable parameters in the LSTM layer
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

print(lstm_params)

1082368


In [14]:
import torch
from torch.utils.data import random_split

# path where .pt features were saved
features_dir = "/kaggle/working/features/train"

# create dataset
full_dataset = PrecomputedFeatureDataset(features_dir)

# split dataset (80% train, 20% validation)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 400
Validation samples: 100


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

for epoch in range(num_epochs):

    # ----- Training -----
    model.train()
    train_loss = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ----- Validation -----
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch [1/10] | Train Loss: 2.1854 | Val Loss: 2.1594 | Val Acc: 0.3200
Epoch [2/10] | Train Loss: 1.9624 | Val Loss: 2.0310 | Val Acc: 0.3200
Epoch [3/10] | Train Loss: 1.8455 | Val Loss: 1.9364 | Val Acc: 0.4400
Epoch [4/10] | Train Loss: 1.7428 | Val Loss: 1.8413 | Val Acc: 0.4700
Epoch [5/10] | Train Loss: 1.6108 | Val Loss: 1.7645 | Val Acc: 0.4900
Epoch [6/10] | Train Loss: 1.5066 | Val Loss: 1.6977 | Val Acc: 0.5000
Epoch [7/10] | Train Loss: 1.4008 | Val Loss: 1.6323 | Val Acc: 0.5800
Epoch [8/10] | Train Loss: 1.3081 | Val Loss: 1.5626 | Val Acc: 0.5700
Epoch [9/10] | Train Loss: 1.2110 | Val Loss: 1.5354 | Val Acc: 0.6000
Epoch [10/10] | Train Loss: 1.1279 | Val Loss: 1.4702 | Val Acc: 0.6200
